# Modelagem e Predição de Scaling Laws (Google Colab)

Este notebook implementa um preditor de leis de escala (Scaling Laws) para redes neurais com base nos dados gerados em nossos experimentos de treinamento.

O objetivo é ajustar duas abordagens de modelagem nos resultados obtidos com os subconjuntos da **Fase 1** para extrapolar e prever deterministicamente a taxa de erro de teste que obteríamos ao treinar no dataset consolidado da **Fase 1 + Fase 2** (8.500 imagens).

### Métodos Utilizados:
1. **Ajuste Teórico (Lei de Rosenfeld)**: Ajuste de curva paramétrico não linear usando a equação empírica:
   $$\varepsilon(M, N) = a \cdot M^{-\alpha} + b \cdot N^{-\beta} + c_\infty$$
   onde $M$ é a fração de capacidade do modelo, $N$ é o tamanho do dataset e $\varepsilon$ é o erro obtido.
2. **Ajuste Empírico (Random Forest Regressor)**: Regressor baseado em árvores que aprende a relação não linear mapeando $[N, M, \text{loss\_slope}]$ para o erro.

## Passo 1: Instalação de Bibliotecas e Configuração

In [ ]:
# Instalação rápida das dependências necessárias
!pip install -q pandas numpy matplotlib scipy scikit-learn

import os
import warnings
from pathlib import Path
from typing import Tuple, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings("ignore")
print("✓ Ambiente pronto e configurado.")

## Passo 2: Equações e Modelos Teóricos

In [ ]:
def rosenfeld_law(X: Tuple[np.ndarray, np.ndarray], a: float, b: float, c_inf: float, alpha: float, beta: float) -> np.ndarray:
    M, N = X
    M_val = np.maximum(M, 1e-5)
    N_val = np.maximum(N, 1e-5)
    return a * (M_val ** -alpha) + b * (N_val ** -beta) + c_inf

def fit_rosenfeld(df: pd.DataFrame) -> Tuple[tuple, bool]:
    M = df["fracao_parametros_modelo"].values
    N = df["tamanho_dataset"].values
    y = df["test_error"].values
    
    unique_points = len(df.groupby(["fracao_parametros_modelo", "tamanho_dataset"]).size())
    
    if unique_points < 5:
        print("[!] Pontos insuficientes para curve_fit. Usando coeficientes padrão de Rosenfeld da literatura.")
        return (0.15, 0.25, 0.02, 0.35, 0.28), False
    
    bounds = (
        (1e-6, 1e-6, 0.0, 1e-4, 1e-4),
        (10.0, 10.0, 1.0, 3.0, 3.0)
    )
    p0 = [0.1, 0.2, 0.01, 0.5, 0.3]
    
    try:
        popt, _ = curve_fit(rosenfeld_law, (M, N), y, p0=p0, bounds=bounds, maxfev=10000)
        return tuple(popt), True
    except Exception as e:
        print(f"[!] curve_fit falhou: {e}. Usando coeficientes padrão.")
        return (0.15, 0.25, 0.02, 0.35, 0.28), False

## Passo 3: Visualização de Superfície de Erro 3D

In [ ]:
def plot_surfaces(df: pd.DataFrame, popt: tuple, rf_model: RandomForestRegressor, mean_loss_slope: float):
    M_grid = np.linspace(0.1, 1.0, 50)
    N_grid = np.linspace(df["tamanho_dataset"].min(), df["tamanho_dataset"].max() * 2, 50)
    MM, NN = np.meshgrid(M_grid, N_grid)
    
    Z_rosenfeld = rosenfeld_law((MM, NN), *popt)
    
    grid_flat_M = MM.flatten()
    grid_flat_N = NN.flatten()
    grid_flat_slope = np.full_like(grid_flat_M, mean_loss_slope)
    X_grid_rf = np.column_stack((grid_flat_N, grid_flat_M, grid_flat_slope))
    Z_rf = rf_model.predict(X_grid_rf).reshape(MM.shape)
    
    fig = plt.figure(figsize=(16, 7))
    
    # Rosenfeld
    ax1 = fig.add_subplot(121, projection='3d')
    surf1 = ax1.plot_surface(MM, NN, Z_rosenfeld, cmap='viridis', alpha=0.8, edgecolor='none')
    ax1.scatter(df["fracao_parametros_modelo"], df["tamanho_dataset"], df["test_error"], color='red', s=50, label='Dados Reais')
    ax1.set_title("Superfície de Erro: Ajuste de Rosenfeld")
    ax1.set_xlabel("Capacidade Modelo (M)")
    ax1.set_ylabel("Tamanho Dataset (N)")
    ax1.set_zlabel("Taxa Erro Teste")
    ax1.legend()
    fig.colorbar(surf1, ax=ax1, shrink=0.5, aspect=10)
    
    # Random Forest
    ax2 = fig.add_subplot(122, projection='3d')
    surf2 = ax2.plot_surface(MM, NN, Z_rf, cmap='magma', alpha=0.8, edgecolor='none')
    ax2.scatter(df["fracao_parametros_modelo"], df["tamanho_dataset"], df["test_error"], color='red', s=50, label='Dados Reais')
    ax2.set_title("Superfície de Erro: Random Forest")
    ax2.set_xlabel("Capacidade Modelo (M)")
    ax2.set_ylabel("Tamanho Dataset (N)")
    ax2.set_zlabel("Taxa Erro Teste")
    ax2.legend()
    fig.colorbar(surf2, ax=ax2, shrink=0.5, aspect=10)
    
    plt.tight_layout()
    plt.show()

## Passo 4: Carregar Métricas e Ajustar Modelos

In [ ]:
# Modifique o caminho abaixo se seu CSV de resultados estiver em outro local no Colab
RESULTS_CSV = "/content/training_results.csv" 

if not os.path.exists(RESULTS_CSV):
    print(f"[!] ERRO: Arquivo {RESULTS_CSV} não encontrado.")
    print("Por favor, execute o notebook train.ipynb primeiro para gerar o CSV, ou faça upload dele.")
else:
    df = pd.read_csv(RESULTS_CSV)
    
    # Ajustar Rosenfeld
    popt, fitted = fit_rosenfeld(df)
    if fitted:
        a, b, c_inf, alpha, beta = popt
        print(f"\n[+] Parâmetros Rosenfeld ajustados:\n  a={a:.4f}, b={b:.4f}, c_inf={c_inf:.4f}, alpha={alpha:.4f}, beta={beta:.4f}")
    else:
        print("\n[*] Usando coeficientes da literatura para Rosenfeld.")
        
    # Ajustar Random Forest
    X_rf = df[["tamanho_dataset", "fracao_parametros_modelo", "loss_slope_final"]].values
    y_rf = df["test_error"].values
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_model.fit(X_rf, y_rf)
    print("[+] Random Forest treinado com sucesso.")

## Passo 5: Projeção Cega (Extrapolação para Fase 1 + Fase 2)

In [ ]:
if 'df' in locals():
    # Parâmetros de Extrapolação
    TARGET_N = 8500  # Fase 1 + Fase 2
    TARGET_M = 1.0   # ResNet-18 com 100% de capacidade
    
    mean_slope = df["loss_slope_final"].mean()
    
    # Predição
    pred_rosenfeld = rosenfeld_law((TARGET_M, TARGET_N), *popt)
    pred_rf = rf_model.predict([[TARGET_N, TARGET_M, mean_slope]])[0]
    
    print("="*50)
    print("             PROJEÇÃO CEGA (EXTRAPOLAÇÃO)           ")
    print("="*50)
    print(f"Alvo da Extrapolação: N={TARGET_N} (Tamanho), M={TARGET_M} (Largura)")
    print(f"Erro de Teste Projetado (Rosenfeld): {pred_rosenfeld * 100:.4f}%")
    print(f"Erro de Teste Projetado (RandomForest): {pred_rf * 100:.4f}%")
    print("="*50)
    
    # Gerar os gráficos de superfície
    plot_surfaces(df, popt, rf_model, mean_slope)

## Passo 6: Validação (Opcional)
Se você já tiver o arquivo de resultados do treinamento real feito em Fase 1 + Fase 2 (`validation_results.csv`), rode esta célula para verificar o desvio do modelo.

In [ ]:
VAL_CSV = "/content/validation_results.csv"

if os.path.exists(VAL_CSV) and 'pred_rosenfeld' in locals():
    val_df = pd.read_csv(VAL_CSV)
    actual_error = val_df["test_error"].mean()
    
    print("="*50)
    print("                 VALIDAÇÃO REAL                   ")
    print("="*50)
    print(f"Erro de Teste Real Observado:        {actual_error * 100:.4f}%")
    print(f"Desvio de Erro (Rosenfeld):          {abs(pred_rosenfeld - actual_error)*100:.4f}%")
    print(f"Desvio de Erro (RandomForest):       {abs(pred_rf - actual_error)*100:.4f}%")
    print("="*50)
else:
    print("Arquivo de validação não encontrado ou predições ausentes. Execute as células anteriores.")